In [ ]:
import os
from google.colab import drive

# 1. 掛載雲端硬碟
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ==========================================
# 2. 安裝環境 (💡 解除舊版封印，完美適應 Colab 新環境)
# ==========================================
%cd /content
if not os.path.exists('/content/yolov7'):
    !git clone https://github.com/ws6125/yolov7.git
%cd yolov7
%pip install -q -r requirements.txt

# 💡 讓 pip 自動安裝支援 NumPy 2.0 的最新版 OpenCV 與 WBF
%pip install -q sahi opencv-python-headless ensemble-boxes

# 3. 複製模型
!cp "/content/drive/MyDrive/實作01pt/best.pt" ./best.pt

import sys
import cv2
import csv
import torch
import numpy as np
import importlib
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion # 💡 載入 WBF 套件

# ==============================================================================
# 1. 自動破解 YOLOv7 的 max_det 限制
# ==============================================================================
general_path = '/content/yolov7/utils/general.py'

with open(general_path, 'r') as f:
    content = f.read()

content = content.replace('max_det = 300', 'max_det = 10000')
content = content.replace('max_nms = 30000', 'max_nms = 100000')

with open(general_path, 'w') as f:
    f.write(content)

if '/content/yolov7' not in sys.path:
    sys.path.append('/content/yolov7')

import utils.general
importlib.reload(utils.general)

from utils.general import scale_coords, xywh2xyxy
from models.experimental import attempt_load
from utils.datasets import letterbox

# ==============================================================================
# 🌟 2. 【新增】專屬 WBF 融合取代函數
# ==============================================================================
def wbf_non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45, img_shape=(3200, 3200), classes=None):
    """
    接收 YOLO 的原始預測張量，轉換並送入 WBF 進行融合
    img_shape: (height, width)
    """
    output = [torch.zeros((0, 6), device=prediction.device)] * prediction.shape[0]

    for xi, x in enumerate(prediction):
        # 1. 基礎信心度過濾
        x = x[x[:, 4] > conf_thres]
        if not x.shape[0]: continue

        # 2. 計算類別分數 = 框信心度 * 類別機率
        x[:, 5:] *= x[:, 4:5]
        conf, j = x[:, 5:].max(1, keepdim=True)

        # 3. 轉為 x1, y1, x2, y2 格式
        x = torch.cat((xywh2xyxy(x[:, :4]), conf, j.float()), 1)[conf.view(-1) > conf_thres]
        if not x.shape[0]: continue

        # 4. 指定類別過濾 (Vehicle)
        if classes is not None:
            x = x[(x[:, 5:6] == torch.tensor(classes, device=x.device)).any(1)]
        if not x.shape[0]: continue

        # 5. 座標正規化 (0~1 之間，WBF 必要條件)
        boxes = x[:, :4].cpu().numpy()
        boxes[:, [0, 2]] /= img_shape[1] # width
        boxes[:, [1, 3]] /= img_shape[0] # height
        boxes = np.clip(boxes, 0.0, 1.0)

        scores = x[:, 4].cpu().numpy()
        labels = x[:, 5].cpu().numpy()

        # 6. 🚀 執行核心 WBF 融合
        boxes_wbf, scores_wbf, labels_wbf = weighted_boxes_fusion(
            [boxes.tolist()], [scores.tolist()], [labels.tolist()],
            weights=None, iou_thr=iou_thres, skip_box_thr=0.0
        )

        # 7. 復原座標比例與 Tensor 格式
        boxes_wbf = torch.tensor(boxes_wbf, device=prediction.device)
        boxes_wbf[:, [0, 2]] *= img_shape[1]
        boxes_wbf[:, [1, 3]] *= img_shape[0]

        scores_wbf = torch.tensor(scores_wbf, device=prediction.device).unsqueeze(1)
        labels_wbf = torch.tensor(labels_wbf, device=prediction.device).unsqueeze(1)

        output[xi] = torch.cat((boxes_wbf, scores_wbf, labels_wbf), 1)

    return output

# ==========================================
# 3. 參數設定區 (多閾值掃描)
# ==========================================
WEIGHTS = './best.pt'
IMAGE_DIR = '/content/drive/MyDrive/testset/images'

IMG_SIZE = 3200
CONF_LIST = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
IOU_THRES = 0.35
TARGET_CLASS = 2     # 車輛
MIN_AREA = 80       # 物理過濾

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 4. 開始推論並批次寫入 CSV
# ==========================================
print("🔄 正在載入 YOLOv7 模型...")
model = attempt_load(WEIGHTS, map_location=device)
model.eval()
stride = int(model.stride.max())

print(f"🚀 開始推論 (已開啟 TTA，並使用 WBF 進行加權融合與多閾值掃描)...")

csv_files = {}
csv_writers = {}
stats = {}

for conf in CONF_LIST:
    filename = f'output_WBF_conf_{conf:.2f}.csv'
    f = open(filename, mode='w', newline='')
    writer = csv.writer(f)
    writer.writerow(["ID", "bbox"])
    csv_files[conf] = f
    csv_writers[conf] = writer
    stats[conf] = {'kept': 0, 'removed': 0}

image_files = sorted([img for img in os.listdir(IMAGE_DIR) if img.lower().endswith(('.png', '.jpg', '.jpeg'))])

for img_name in tqdm(image_files):
    img_path = os.path.join(IMAGE_DIR, img_name)
    img0 = cv2.imread(img_path)
    if img0 is None: continue

    img = letterbox(img0, IMG_SIZE, stride=stride, auto=False)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)

    img_tensor = torch.from_numpy(img).to(device).float() / 255.0
    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    # 執行推論 (TTA)
    with torch.no_grad():
        pred_raw = model(img_tensor, augment=True)[0]

    img_shape = img_tensor.shape[2:]

    # 💡 針對不同的 Confidence 門檻，分別呼叫 WBF 並寫入 CSV
    for conf_thres in CONF_LIST:

        # 呼叫自定義的 WBF 函數
        pred = wbf_non_max_suppression(
            pred_raw.clone(), # 必須 clone，否則會修改到原始預測
            conf_thres=conf_thres,
            iou_thres=IOU_THRES,
            img_shape=img_shape,
            classes=[TARGET_CLASS]
        )

        crd = []
        for det in pred:
            if len(det):
                # 將座標放大回原圖尺寸
                det[:, :4] = scale_coords(img_shape, det[:, :4], img0.shape).round()

                for *xyxy, conf_score, cls in det:
                    x_min, y_min = int(xyxy[0]), int(xyxy[1])
                    x_max, y_max = int(xyxy[2]), int(xyxy[3])
                    area = (x_max - x_min) * (y_max - y_min)

                    if area < MIN_AREA:
                        stats[conf_thres]['removed'] += 1
                        continue

                    stats[conf_thres]['kept'] += 1
                    crd.append(f"{x_min} {y_min} {x_max} {y_max}")

        bbox_string = " ".join(crd) if len(crd) > 0 else "0"
        csv_writers[conf_thres].writerow([img_name, bbox_string])

# 關閉所有檔案
for f in csv_files.values():
    f.close()

print(f"\n✅ 全部完成！限制已解除，TTA 開啟，WBF 融合已完美套用！")
print("📊 WBF 各信心度閾值 (Confidence Threshold) 成果統計：")
for conf in CONF_LIST:
    print(f"  ➤ Conf: {conf:.2f} | 保留車輛: {stats[conf]['kept']} 個 | 過濾雜訊: {stats[conf]['removed']} 個 | 輸出檔: output_WBF_conf_{conf:.2f}.csv")

Mounted at /content/drive
/content
Cloning into 'yolov7'...
remote: Enumerating objects: 943, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 943 (delta 12), reused 9 (delta 9), pack-reused 922 (from 2)
Receiving objects: 100% (943/943), 57.73 MiB | 22.80 MiB/s, done.
Resolving deltas: 100% (420/420), done.
/content/yolov7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 4.21.2 which is incompatible.
google-cloud-bigtable 2.38.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.21.2 which 

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


🚀 開始推論 (已開啟 TTA，並使用 WBF 進行加權融合與多閾值掃描)...


100%|██████████| 53/53 [03:39<00:00,  4.15s/it]


✅ 全部完成！限制已解除，TTA 開啟，WBF 融合已完美套用！
📊 WBF 各信心度閾值 (Confidence Threshold) 成果統計：
  ➤ Conf: 0.50 | 保留車輛: 14092 個 | 過濾雜訊: 4 個 | 輸出檔: output_WBF_conf_0.50.csv
  ➤ Conf: 0.55 | 保留車輛: 12391 個 | 過濾雜訊: 2 個 | 輸出檔: output_WBF_conf_0.55.csv
  ➤ Conf: 0.60 | 保留車輛: 10652 個 | 過濾雜訊: 0 個 | 輸出檔: output_WBF_conf_0.60.csv
  ➤ Conf: 0.65 | 保留車輛: 8711 個 | 過濾雜訊: 0 個 | 輸出檔: output_WBF_conf_0.65.csv
  ➤ Conf: 0.70 | 保留車輛: 6679 個 | 過濾雜訊: 0 個 | 輸出檔: output_WBF_conf_0.70.csv
  ➤ Conf: 0.75 | 保留車輛: 4470 個 | 過濾雜訊: 0 個 | 輸出檔: output_WBF_conf_0.75.csv
  ➤ Conf: 0.80 | 保留車輛: 2465 個 | 過濾雜訊: 0 個 | 輸出檔: output_WBF_conf_0.80.csv
